# Cuantización MLP27K Montmeló para FINN

Este notebook parte del modelo bueno encontrado para el dataset de Montmeló:

- **Checkpoint:** `mlp_27k.ckpt`
- **Arquitectura:** `576 → 32 → 256 → 2`
- **Entrada:** imagen RGB 16×12 en formato plano `RRR...GGG...BBB`, normalizada a `[0,1]`
- **Salida:** dos valores normalizados `[0,1]`, correspondientes a `v` y `w`

Objetivo del notebook:

1. Cargar `mlp_27k.ckpt`.
2. Crear una MLP equivalente cuantizada con Brevitas.
3. Copiar los pesos del checkpoint bueno.
4. Exportar un ONNX cuantizado compatible con FINN.
5. Comprobarlo contra el CSV del dataset.
6. Intentar ejecutar `build_dataflow` para generar hardware FINN.

Este notebook **no usa** la red antigua `BobEsponja`, ni `TopK`, ni clasificación de 14 clases.

## 1. Imports y rutas

Antes de ejecutar, asegúrate de que en la misma carpeta del notebook están estos ficheros:

```text
mlp_27k.ckpt
```

Opcionalmente también puede estar:

```text
mlp27k_clean_inferred.onnx
```

El notebook buscará automáticamente el dataset en varias rutas habituales.

In [1]:
import os
import csv
import math
import json
import shutil
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn

import brevitas.nn as qnn
import brevitas.onnx as bo
from brevitas.quant import Int8Bias

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.onnx_exec import execute_onnx
from qonnx.core.datatype import DataType
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.general import (
    GiveReadableTensorNames,
    GiveUniqueNodeNames,
    RemoveStaticGraphInputs,
    RemoveUnusedTensors,
)

print('Torch:', torch.__version__)
print('Trabajo en:', Path.cwd())

Torch: 1.7.1
Trabajo en: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example


In [2]:
# Rutas principales
CKPT_PATH = Path('mlp_27k.ckpt')
QONNX_OUT = Path('mlp27k_quantized.onnx')
QONNX_TIDY = Path('mlp27k_quantized_tidy.onnx')
MODEL_WORK = Path('model.onnx')
FINAL_OUTPUT_DIR = 'output_final_mlp27k_quantized'

# FINN necesita esta variable para guardar modelos intermedios.
# Usamos /tmp porque normalmente no da problemas de permisos dentro del Docker.
os.environ['FINN_BUILD_DIR'] = '/tmp/FINN_build_mlp27k_quantized'
Path(os.environ['FINN_BUILD_DIR']).mkdir(parents=True, exist_ok=True)

print('CKPT_PATH:', CKPT_PATH.resolve())
print('QONNX_OUT:', QONNX_OUT.resolve())
print('FINN_BUILD_DIR:', os.environ['FINN_BUILD_DIR'])
print('FINAL_OUTPUT_DIR:', FINAL_OUTPUT_DIR)

assert CKPT_PATH.exists(), f'No existe {CKPT_PATH}. Copia mlp_27k.ckpt a esta carpeta.'

CKPT_PATH: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/mlp_27k.ckpt
QONNX_OUT: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/mlp27k_quantized.onnx
FINN_BUILD_DIR: /tmp/FINN_build_mlp27k_quantized
FINAL_OUTPUT_DIR: output_final_mlp27k_quantized


## 2. Modelo cuantizado Brevitas

Esta red mantiene la arquitectura del checkpoint bueno:

```text
576 → 32 → 256 → 2
```

La idea es copiar los pesos buenos y exportar a ONNX FINN/QONNX.

In [3]:
class QuantMLP27K(nn.Module):
    def __init__(self):
        super().__init__()

        # Entrada normalizada 0-1.
        self.quant_in = qnn.QuantIdentity(
            bit_width=8,
            return_quant_tensor=True
        )

        self.fc1 = qnn.QuantLinear(
            576,
            32,
            bias=True,
            weight_bit_width=8,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )
        self.relu1 = qnn.QuantReLU(
            bit_width=8,
            return_quant_tensor=True
        )

        self.fx = qnn.QuantLinear(
            32,
            256,
            bias=True,
            weight_bit_width=8,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )
        self.relu2 = qnn.QuantReLU(
            bit_width=8,
            return_quant_tensor=True
        )

        self.predictor = qnn.QuantLinear(
            256,
            2,
            bias=True,
            weight_bit_width=8,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

        # Salida cuantizada. La red buena trabaja con salida 0-1.
        self.quant_out = qnn.QuantIdentity(
            bit_width=8,
            return_quant_tensor=True
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)

        x = self.quant_in(x)

        x = self.fc1(x)
        x = self.relu1(x)

        x = self.fx(x)
        x = self.relu2(x)

        x = self.predictor(x)
        x = self.quant_out(x)

        # Para dejar salida tensor normal en PyTorch.
        if hasattr(x, 'value'):
            x = x.value
        elif isinstance(x, tuple):
            x = x[0]

        return x

qmodel = QuantMLP27K()
qmodel.eval()
print(qmodel)

QuantMLP27K(
  (quant_in): QuantIdentity(
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (act_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (fused_activation_quant_proxy): FusedActivationQuantProxy(
        (activation_impl): Identity()
        (tensor_quant): RescalingIntQuant(
          (int_quant): IntQuant(
            (float_to_int_impl): RoundSte()
            (tensor_clamp_impl): TensorClamp()
            (delay_wrapper): DelayWrapper(
              (delay_impl): _NoDelay()
            )
          )
          (scaling_impl): ParameterFromRuntimeStatsScaling(
            (stats_input_view_shape_impl): OverTensorView()
            (stats): _Stats(
              (stats_impl): AbsPercentile()
            )
            (restrict_clamp_scaling): _RestrictClampValue(
              (clamp_min_ste): Identity()
              (restrict_value_impl): FloatRestrictValue()
            )
            (r

## 3. Cargar los pesos de `mlp_27k.ckpt`

El checkpoint puede traer prefijos tipo `model.` o `predictor.0.`. Esta celda los limpia y carga únicamente los tensores compatibles.

In [5]:
def extract_state_dict(obj):
    if isinstance(obj, dict):
        for key in ['state_dict', 'model_state_dict', 'net', 'model']:
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
        return obj
    return obj.state_dict()


def clean_state_dict(sd):
    out = {}

    for k, v in sd.items():
        if not torch.is_tensor(v):
            continue

        nk = k

        changed = True
        while changed:
            changed = False
            for pref in ['model.', 'base_model.', 'module.', 'net.']:
                if nk.startswith(pref):
                    nk = nk[len(pref):]
                    changed = True

        nk = nk.replace('predictor.0.', 'predictor.')
        out[nk] = v

    return out


obj = torch.load(str(CKPT_PATH), map_location='cpu')
sd = clean_state_dict(extract_state_dict(obj))

print('Tensores encontrados en checkpoint:')
for k, v in sd.items():
    if torch.is_tensor(v):
        print(k, tuple(v.shape))

Tensores encontrados en checkpoint:
fc1.weight (32, 576)
fc1.bias (32,)
fx.weight (256, 32)
fx.bias (256,)
predictor.weight (2, 256)
predictor.bias (2,)


In [6]:
qmodel = QuantMLP27K()
qmodel_sd = qmodel.state_dict()

loaded = []
not_loaded = []

for k, v in sd.items():
    if k in qmodel_sd and tuple(qmodel_sd[k].shape) == tuple(v.shape):
        qmodel_sd[k] = v
        loaded.append(k)
    else:
        not_loaded.append(k)

qmodel.load_state_dict(qmodel_sd, strict=False)
qmodel.eval()

print('Pesos cargados:')
for k in loaded:
    print(' ', k)

print('Total cargados:', len(loaded))
print('No cargados:', not_loaded)

assert len(loaded) >= 6, 'No se han cargado los 6 pesos principales. Revisa arquitectura/checkpoint.'

Pesos cargados:
  fc1.weight
  fc1.bias
  fx.weight
  fx.bias
  predictor.weight
  predictor.bias
Total cargados: 6
No cargados: []


## 4. Buscar el dataset y preparar funciones de comparación

Se comparará el modelo cuantizado contra el CSV del dataset, igual que hicimos con el ONNX limpio.

In [7]:
def find_dataset_dir():
    candidates = [
        Path('DATA_12_16/counter_montmelo_holonomic'),
        Path('/tmp/Orig/DATA_12_16/counter_montmelo_holonomic'),
        Path('/tmp/Orig/Orig/DATA_12_16/counter_montmelo_holonomic'),
        Path('/home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic'),
        Path('/home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/output_final_miguel/Dataset_JdeRobot/DATA/counter_montmelo_holonomic'),
    ]

    for d in candidates:
        if (d / 'data.csv').exists():
            return d

    raise FileNotFoundError('No encuentro el data.csv del dataset. Copia DATA_12_16 o ajusta la ruta.')


DATASET_DIR = find_dataset_dir()
CSV_PATH = DATASET_DIR / 'data.csv'

print('Dataset:', DATASET_DIR)
print('CSV:', CSV_PATH)


def image_to_flat_0_1(img_path):
    img = Image.open(img_path).convert('RGB')
    img = img.resize((16, 12))
    arr = np.asarray(img).astype(np.float32)
    # HWC -> CHW -> RRR GGG BBB
    arr = np.transpose(arr, (2, 0, 1)).reshape(-1)
    arr = arr / 255.0
    return torch.tensor(arr, dtype=torch.float32).reshape(1, 576)


def image_to_flat_uint8(img_path):
    img = Image.open(img_path).convert('RGB')
    img = img.resize((16, 12))
    arr = np.asarray(img).astype(np.float32)
    arr = np.transpose(arr, (2, 0, 1)).reshape(-1)
    return arr.reshape(1, 576).astype(np.float32)


def denorm_v(vnorm):
    return 13.6 * vnorm - 0.6


def denorm_w(wnorm):
    return 2.0 * math.pi * wnorm - math.pi


def norm_v(v):
    return (v + 0.6) / 13.6


def norm_w(w):
    return (w + math.pi) / (2.0 * math.pi)

Dataset: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic
CSV: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic/data.csv


## 5. Comprobar el modelo cuantizado en PyTorch contra el CSV

Esta prueba se hace antes de exportar a ONNX. Si aquí ya sale muy mal, habría que hacer fine-tuning cuantizado.

In [9]:
def eval_torch_model(model, n_samples=2000, verbose_first=10):
    model.eval()

    err_v = []
    err_w = []
    pred_v = []
    pred_w = []

    n = 0

    with torch.no_grad():
        with open(CSV_PATH) as f:
            reader = csv.DictReader(f)

            for row in reader:
                img_path = DATASET_DIR / row['image_name']
                if not img_path.exists():
                    continue

                x = image_to_flat_0_1(img_path)
                y = model(x).reshape(-1).detach().cpu().numpy()

                vnorm = float(y[0])
                wnorm = float(y[1])

                v_pred = denorm_v(vnorm)
                w_pred = denorm_w(wnorm)

                v_csv = float(row['v'])
                w_csv = float(row['w'])

                err_v.append(v_pred - v_csv)
                err_w.append(w_pred - w_csv)
                pred_v.append(v_pred)
                pred_w.append(w_pred)

                if n < verbose_first:
                    raw_v_csv = round(norm_v(v_csv) * 255)
                    raw_w_csv = round(norm_w(w_csv) * 255)
                    raw_v_pred = round(vnorm * 255)
                    raw_w_pred = round(wnorm * 255)
                    print(
                        f"{row['image_name']} | "
                        f"CSV v={v_csv:.4f} w={w_csv:.4f} raw_csv={raw_v_csv},{raw_w_csv} | "
                        f"QPyTorch v={v_pred:.4f} w={w_pred:.4f} raw={raw_v_pred},{raw_w_pred}"
                    )

                n += 1
                if n >= n_samples:
                    break

    err_v = np.array(err_v)
    err_w = np.array(err_w)
    pred_v = np.array(pred_v)
    pred_w = np.array(pred_w)

    print('Frames:', n)
    print('MAE_v =', np.mean(np.abs(err_v)))
    print('MAE_w =', np.mean(np.abs(err_w)))
    print('v pred min/max/std =', pred_v.min(), pred_v.max(), pred_v.std())
    print('w pred min/max/std =', pred_w.min(), pred_w.max(), pred_w.std())

    return {
        'frames': n,
        'mae_v': float(np.mean(np.abs(err_v))),
        'mae_w': float(np.mean(np.abs(err_w))),
        'std_v': float(pred_v.std()),
        'std_w': float(pred_w.std()),
    }


metrics_torch = eval_torch_model(qmodel, n_samples=2000)
metrics_torch

1.png | CSV v=5.0000 w=-0.2254 raw_csv=105,118 | QPyTorch v=0.2500 w=-3.0434 raw=16,4
2.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | QPyTorch v=0.2500 w=-3.0434 raw=16,4
3.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | QPyTorch v=0.2500 w=-3.0434 raw=16,4
4.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | QPyTorch v=0.2500 w=-3.0434 raw=16,4
5.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | QPyTorch v=0.2500 w=-3.0434 raw=16,4
6.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | QPyTorch v=0.2500 w=-3.0434 raw=16,4
7.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | QPyTorch v=0.2500 w=-3.0434 raw=16,4
8.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | QPyTorch v=0.2500 w=-3.0434 raw=16,4
9.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | QPyTorch v=0.2500 w=-3.0434 raw=16,4
10.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | QPyTorch v=0.2500 w=-3.0434 raw=16,4
Frames: 2000
MAE_v = 4.86990625
MAE_w = 2.970138150256047
v pred min/max/std = -1.13125 2.1624999999999996 0.3009181392396236
w pred min/ma

{'frames': 2000,
 'mae_v': 4.86990625,
 'mae_w': 2.970138150256047,
 'std_v': 0.3009181392396236,
 'std_w': 0.20439928259748094}

## 6. Exportar a ONNX FINN/QONNX cuantizado

Si esta celda falla, revisa versión de Brevitas/FINN. Si exporta bien, se generará:

```text
mlp27k_quantized.onnx
```

In [ ]:
qmodel.eval()

dummy_input = torch.rand(1, 576)

bo.export_finn_onnx(
    qmodel,
    input_shape=(1, 576),
    export_path=str(QONNX_OUT)
)

print('ONNX cuantizado exportado:', QONNX_OUT)
!ls -lh mlp27k_quantized.onnx

## 7. Tidy-up del ONNX cuantizado

Se aplican transformaciones básicas y se guarda como:

```text
mlp27k_quantized_tidy.onnx
model.onnx
```

In [ ]:
model = ModelWrapper(str(QONNX_OUT))

model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())
model = model.transform(RemoveUnusedTensors())

model.save(str(QONNX_TIDY))
shutil.copyfile(QONNX_TIDY, MODEL_WORK)

print('Guardado:', QONNX_TIDY)
print('Copiado como:', MODEL_WORK)

m = ModelWrapper(str(MODEL_WORK))
print('Input:')
for inp in m.graph.input:
    print(inp.name, m.get_tensor_shape(inp.name), m.get_tensor_datatype(inp.name))
print('Output:')
for out in m.graph.output:
    print(out.name, m.get_tensor_shape(out.name), m.get_tensor_datatype(out.name))
print('Nodos:')
for node in m.graph.node:
    print(node.name, node.op_type)

## 8. Comprobar el ONNX cuantizado contra el CSV

Aquí se ejecuta `model.onnx` con QONNX y se compara contra el CSV.

In [ ]:
def eval_onnx_model(onnx_path='model.onnx', n_samples=2000, verbose_first=10):
    model = ModelWrapper(onnx_path)
    model = model.transform(InferShapes())

    input_name = model.graph.input[0].name
    output_name = model.graph.output[0].name

    print('Input:', input_name, model.get_tensor_shape(input_name), model.get_tensor_datatype(input_name))
    print('Output:', output_name, model.get_tensor_shape(output_name), model.get_tensor_datatype(output_name))

    err_v = []
    err_w = []
    pred_v = []
    pred_w = []

    n = 0

    with open(CSV_PATH) as f:
        reader = csv.DictReader(f)

        for row in reader:
            img_path = DATASET_DIR / row['image_name']
            if not img_path.exists():
                continue

            # El modelo cuantizado exportado espera entrada 0-1 salvo que se le añada preprocesado /255 aparte.
            x = image_to_flat_0_1(img_path).numpy().astype(np.float32)

            y_dict = execute_onnx(model, {input_name: x})
            y = y_dict[output_name].reshape(-1)

            vnorm = float(y[0])
            wnorm = float(y[1])

            v_pred = denorm_v(vnorm)
            w_pred = denorm_w(wnorm)

            v_csv = float(row['v'])
            w_csv = float(row['w'])

            err_v.append(v_pred - v_csv)
            err_w.append(w_pred - w_csv)
            pred_v.append(v_pred)
            pred_w.append(w_pred)

            if n < verbose_first:
                raw_v_csv = round(norm_v(v_csv) * 255)
                raw_w_csv = round(norm_w(w_csv) * 255)
                raw_v_pred = round(vnorm * 255)
                raw_w_pred = round(wnorm * 255)
                print(
                    f"{row['image_name']} | "
                    f"CSV v={v_csv:.4f} w={w_csv:.4f} raw_csv={raw_v_csv},{raw_w_csv} | "
                    f"QONNX v={v_pred:.4f} w={w_pred:.4f} raw={raw_v_pred},{raw_w_pred}"
                )

            n += 1
            if n >= n_samples:
                break

    err_v = np.array(err_v)
    err_w = np.array(err_w)
    pred_v = np.array(pred_v)
    pred_w = np.array(pred_w)

    print('
Frames:', n)
    print('MAE_v =', np.mean(np.abs(err_v)))
    print('MAE_w =', np.mean(np.abs(err_w)))
    print('v pred min/max/std =', pred_v.min(), pred_v.max(), pred_v.std())
    print('w pred min/max/std =', pred_w.min(), pred_w.max(), pred_w.std())

    return {
        'frames': n,
        'mae_v': float(np.mean(np.abs(err_v))),
        'mae_w': float(np.mean(np.abs(err_w))),
        'std_v': float(pred_v.std()),
        'std_w': float(pred_w.std()),
    }


metrics_onnx = eval_onnx_model('model.onnx', n_samples=2000)
metrics_onnx

## 9. Añadir preprocesado `/255` dentro del ONNX, opcional

El testbench de Verilator normalmente envía bytes `0–255`. Para que el hardware reciba bytes y el modelo trabaje internamente en `0–1`, añadimos un preprocesado tipo `ToTensor`, que divide por 255.

Si esta celda falla, no pasa nada: se puede seguir con el modelo sin preprocesado y adaptar el testbench, pero lo ideal es meter `/255` dentro del ONNX.

In [ ]:
try:
    from finn.util.pytorch import ToTensor
    from qonnx.transformation.merge_onnx_models import MergeONNXModels

    BASE_QMODEL = Path('model.onnx')
    PREPROC_ONNX = Path('preproc_div255_mlp27k_quant.onnx')
    MODEL_FOR_FINN_UINT8 = Path('model_for_finn_mlp27k_quant_uint8.onnx')

    model = ModelWrapper(str(BASE_QMODEL))
    model = model.transform(InferShapes())

    global_inp_name = model.graph.input[0].name
    ishape = model.get_tensor_shape(global_inp_name)
    print('Shape base:', ishape)

    totensor_pyt = ToTensor()
    bo.export_finn_onnx(totensor_pyt, ishape, str(PREPROC_ONNX))

    pre_model = ModelWrapper(str(PREPROC_ONNX))
    merged = model.transform(MergeONNXModels(pre_model))

    global_inp_name = merged.graph.input[0].name
    merged.set_tensor_datatype(global_inp_name, DataType['UINT8'])

    merged = merged.transform(InferShapes())
    merged = merged.transform(FoldConstants())
    merged = merged.transform(GiveUniqueNodeNames())
    merged = merged.transform(GiveReadableTensorNames())
    merged = merged.transform(InferDataTypes())
    merged = merged.transform(RemoveStaticGraphInputs())
    merged = merged.transform(RemoveUnusedTensors())

    merged.save(str(MODEL_FOR_FINN_UINT8))
    shutil.copyfile(MODEL_FOR_FINN_UINT8, MODEL_WORK)

    print('Modelo con entrada UINT8 guardado en:', MODEL_FOR_FINN_UINT8)
    print('También copiado como:', MODEL_WORK)

except Exception as e:
    print('No se pudo añadir preprocesado /255 dentro del ONNX.')
    print('Error:', repr(e))
    print('Se mantiene model.onnx sin preprocesado interno.')

## 10. Comprobar `model.onnx` tras el preprocesado opcional

Si el preprocesado `/255` se añadió bien, ahora `model.onnx` espera entrada `UINT8`. Esta celda ejecuta con imágenes `0–255`.

In [ ]:
def eval_onnx_model_current_input(onnx_path='model.onnx', n_samples=2000, verbose_first=10):
    model = ModelWrapper(onnx_path)
    model = model.transform(InferShapes())

    input_name = model.graph.input[0].name
    output_name = model.graph.output[0].name
    input_dtype = model.get_tensor_datatype(input_name)

    print('Input:', input_name, model.get_tensor_shape(input_name), input_dtype)
    print('Output:', output_name, model.get_tensor_shape(output_name), model.get_tensor_datatype(output_name))

    err_v = []
    err_w = []
    pred_v = []
    pred_w = []

    n = 0

    with open(CSV_PATH) as f:
        reader = csv.DictReader(f)

        for row in reader:
            img_path = DATASET_DIR / row['image_name']
            if not img_path.exists():
                continue

            # Si la entrada es UINT8, usamos valores 0-255.
            # Si no, usamos 0-1.
            if str(input_dtype) == 'UINT8':
                x = image_to_flat_uint8(img_path)
            else:
                x = image_to_flat_0_1(img_path).numpy().astype(np.float32)

            y_dict = execute_onnx(model, {input_name: x})
            y = y_dict[output_name].reshape(-1)

            vnorm = float(y[0])
            wnorm = float(y[1])

            v_pred = denorm_v(vnorm)
            w_pred = denorm_w(wnorm)

            v_csv = float(row['v'])
            w_csv = float(row['w'])

            err_v.append(v_pred - v_csv)
            err_w.append(w_pred - w_csv)
            pred_v.append(v_pred)
            pred_w.append(w_pred)

            if n < verbose_first:
                raw_v_csv = round(norm_v(v_csv) * 255)
                raw_w_csv = round(norm_w(w_csv) * 255)
                raw_v_pred = round(vnorm * 255)
                raw_w_pred = round(wnorm * 255)
                print(
                    f"{row['image_name']} | "
                    f"CSV v={v_csv:.4f} w={w_csv:.4f} raw_csv={raw_v_csv},{raw_w_csv} | "
                    f"QONNX v={v_pred:.4f} w={w_pred:.4f} raw={raw_v_pred},{raw_w_pred}"
                )

            n += 1
            if n >= n_samples:
                break

    err_v = np.array(err_v)
    err_w = np.array(err_w)
    pred_v = np.array(pred_v)
    pred_w = np.array(pred_w)

    print('
Frames:', n)
    print('MAE_v =', np.mean(np.abs(err_v)))
    print('MAE_w =', np.mean(np.abs(err_w)))
    print('v pred min/max/std =', pred_v.min(), pred_v.max(), pred_v.std())
    print('w pred min/max/std =', pred_w.min(), pred_w.max(), pred_w.std())

    return {
        'frames': n,
        'mae_v': float(np.mean(np.abs(err_v))),
        'mae_w': float(np.mean(np.abs(err_w))),
        'std_v': float(pred_v.std()),
        'std_w': float(pred_w.std()),
    }


metrics_current = eval_onnx_model_current_input('model.onnx', n_samples=2000)
metrics_current

## 11. Intentar build_dataflow de FINN

Esta celda intenta generar el hardware FINN. Si falla, leer el error:

- Si falla en `StreamingDataflowPartition`, todavía hay operaciones sin convertir a HLS.
- Si falla en HLS/IPGen, ya habría que depurar la generación de IP.

In [ ]:
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg

# Borrar salida anterior para no mezclar resultados.
if Path(FINAL_OUTPUT_DIR).exists():
    shutil.rmtree(FINAL_OUTPUT_DIR)
    print('Borrado output anterior:', FINAL_OUTPUT_DIR)

cfg = build_cfg.DataflowBuildConfig(
    output_dir=FINAL_OUTPUT_DIR,
    folding_config_file=None,
    auto_fifo_depths=True,
    mvau_wwidth_max=36,
    target_fps=100,
    synth_clk_period_ns=10.0,
    fpga_part='xc7z020clg400-1',
    board=None,
    stitched_ip_gen_dcp=False,
    default_mem_mode='const',
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
        build_cfg.DataflowOutputType.STITCHED_IP,
        build_cfg.DataflowOutputType.RTLSIM_PERFORMANCE,
        build_cfg.DataflowOutputType.OOC_SYNTH,
    ]
)

print('FINN_BUILD_DIR:', os.environ['FINN_BUILD_DIR'])
print('Lanzando build_dataflow con:', MODEL_WORK)
print('Output dir:', FINAL_OUTPUT_DIR)

try:
    build.build_dataflow_cfg(str(MODEL_WORK), cfg)
    print('FINN build terminado')
except Exception as e:
    print('FINN build ha fallado.')
    print('Error:', repr(e))
    print('Mira el log con la siguiente celda.')
    raise

## 12. Inspección si falla FINN

Ejecuta estas celdas si el build falla.

In [ ]:
# Últimas líneas del log del build
log_path = Path(FINAL_OUTPUT_DIR) / 'build_dataflow.log'
print('Log:', log_path)
if log_path.exists():
    !tail -120 {log_path}
else:
    print('No existe todavía el log')

In [ ]:
# Listar ONNX intermedios generados por FINN
build_dir = Path(os.environ['FINN_BUILD_DIR'])
onnx_files = sorted(build_dir.rglob('*.onnx'))

print('ONNX encontrados:', len(onnx_files))
for i, f in enumerate(onnx_files):
    print(i, f)

In [ ]:
# Inspeccionar el último ONNX intermedio
if len(onnx_files) > 0:
    last_model = onnx_files[-1]
    print('Inspeccionando:', last_model)

    m = ModelWrapper(str(last_model))

    print('Inputs:')
    for inp in m.graph.input:
        print(inp.name, m.get_tensor_shape(inp.name), m.get_tensor_datatype(inp.name))

    print('Outputs:')
    for out in m.graph.output:
        print(out.name, m.get_tensor_shape(out.name), m.get_tensor_datatype(out.name))

    print('Nodos:')
    for node in m.graph.node:
        print(node.name, node.op_type)
else:
    print('No hay modelos ONNX intermedios')

## 13. Conclusión esperada

Si la cuantización rápida funciona bien contra el CSV y FINN genera `stitched_ip`, entonces el siguiente paso será repetir la simulación Verilator y volver a comparar contra el CSV.

Si la cuantización rápida empeora mucho el error, habrá que hacer **fine-tuning cuantizado**: entrenar unas épocas el modelo Brevitas cuantizado usando el CSV, partiendo de los pesos de `mlp_27k.ckpt`.